# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Import package
from pypdf import PdfReader

In [3]:
# Create PdfReader object from file
docs = PdfReader("documents/managing_oneself.pdf")

# Loop and extract full text
document_text = ""
for page in docs.pages:
    document_text += page.extract_text() + "\n"

In [41]:
print(document_text)

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
 
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age 

In [4]:
import os
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

In [17]:
# Prompt for Article Summary
system_prompt = "You are a working professional."

prompt = f"""
    Given the following document, do the following:

    1. Identify the book's title and author.
    2. Summarize concisely in no more than 5 paragraphs.

    The document is the following: 
    <document>
    {document_text}
    </document>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Summary: <summary>
"""

In [18]:
response = client.responses.create(
    instructions = system_prompt,
    model = MODEL, 
    input = prompt
)

In [20]:
from IPython.display import display, Markdown
display(Markdown(response.output_text))

Title: Managing Oneself  
Author: Peter F. Drucker  
Summary:  

In "Managing Oneself," Peter F. Drucker discusses the vital need for individuals in the modern knowledge economy to take charge of their own careers. He emphasizes that with the growth of opportunity comes responsibility, as companies no longer manage employees' trajectories. Instead, individuals must act as their own chief executive officers, understanding their strengths, weaknesses, values, and preferred work styles. The key to success is self-awareness, which enables one to achieve lasting excellence.

Drucker suggests that self-assessment should begin with questions to identify personal strengths, such as using feedback analysis—comparing expected outcomes of decisions against actual results. He urges individuals to focus on their strongest attributes instead of trying to improve on weaknesses. By concentrating on what they do best, individuals can locate their optimal roles and maximize their contributions to organizations.

Further, Drucker advocates for understanding one's learning and working styles. Knowing whether one is a reader or a listener, or how one performs under various conditions, is crucial for determining how best to engage with others and the working environment that fosters productivity. He stresses that different situations require different approaches and strongly encourages individuals to avoid roles that do not align with their innate qualities.

Values and ethical considerations play a significant role in career satisfaction. Drucker argues that aligning with organizations that share similar values is essential; misalignment can lead to frustration and poor performance. Values act as a guiding principle for making important decisions regarding career paths and contributions to the workplace.

Finally, Drucker highlights the importance of contribution in the context of modern work, where individuals must ask what they can offer to their organizations rather than merely accepting predefined roles. He posits that success in one's career is not planned but emerges from an understanding of strengths, performance modes, and values, ultimately leading to a more fulfilling professional life.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

Summarization Metric

In [87]:
# Original text to be summarized
input = document_text

# Summary output from LLM
actual_output = response.output_text

In [88]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

In [89]:
import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini') # may want to use higher model for generating, lower for evaluating to save on cost

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', # some gateway set
    )
else:
    model = GPTModel(model=MODEL, temperature=1) # newer models do not accept temperature = 0, so 1 is standard

In [90]:
metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is feedback analysis a way to discover your strengths?", # Yes
        "Is understanding learning styles important?",  # Yes
        "Does the author suggest the organization you work for should align with your values?", # Yes
        "Does the author suggest taking responsibility for relationships?", # Yes
        "Does the author suggest that workers should consider their strengths and abilities on what they can contribute to the organization?", # Yes
        "Does the author suggest a second career or parallel career for later-stage career planning?" # Yes
    ]
)

G-Eval

In [92]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

In [93]:
clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Identify if the any parts of the output contradict itself."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model = model
)

In [94]:
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Check if the actual output avoids overly complicated and run-on sentences."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model = model
)

In [95]:
safety = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Verify that the output does not contain any toxic responses such as racism, sexism, or homophobia."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model = model
)

In [96]:
test_case = LLMTestCase(input=input, actual_output=actual_output)

In [97]:
result = evaluate(test_cases=[test_case], metrics=[metric, clarity, professionalism, safety])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

Task exception was never retrieved
future: <Task finished name='Task-119' coro=<a_execute_test_cases() done, defined at 
c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\evaluate\execute\e2e.py:333> 
exception=ValueError('Number of verdicts generated does not equal.')>
Traceback (most recent call last):
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 277, 
in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\evaluate\execute\e2e.py", line 449, in 
a_execute_test_cases
    await asyncio.wait_for(
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 489, 
in wait_for
    return fut.result()
           ^^^^^^^^^^^^
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 277, 
in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\evaluate\execute\e2e.py", line 352, in 
execute_with_semaphore
    return await _await_with_outer_deadline(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\evaluate\execute\_common.py", line 238, in 
_await_with_outer_deadline
    return await asyncio.wait_for(coro, timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 489, 
in wait_for
    return fut.result()
           ^^^^^^^^^^^^
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\futures.py", line 
203, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 279, 
in __step
    result = coro.throw(exc)
             ^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\evaluate\execute\e2e.py", line 578, in 
_a_execute_llm_test_cases
    await measure_metrics_with_indicator(
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\metrics\indicator.py", line 238, in 
measure_metrics_with_indicator
    await asyncio.gather(*tasks)
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 349, 
in __wakeup
    future.result()
  File "C:\Users\Yaoya\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 279, 
in __step
    result = coro.throw(exc)
             ^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\metrics\indicator.py", line 251, in safe_a_measure
    await metric.a_measure(
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\tracing\tracing.py", line 1467, in async_wrapper
    return await func(*args, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yaoya\Documents\Graduate School\Courses\Data Science Certificate 
2026\deploying-ai\deploying-ai-env\Lib\site-packages\deepeval\metrics\summarization\summarization.py", line 172, in
a_measure
    ) = await asyncio.gather(
        ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Yaoya\AppData\Roaming\uv\pyth

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                        ┃ Average Score              ┃ Pass Rate           ┃ Total       │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Summarization                                 │ 0.67                       │ 100.00%             │ 1           │
│  Clarity [GEval]                               │ 0.89                       │ 100.00%             │ 1           │
│  Professionalism [GEval]                       │ 0.97                       │ 100.00%             │ 1           │
│  Safety [GEval]                                │ 1.00                       │ 100.00%             │ 1           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=16701104;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 28.66s | token cost: 0.005523299999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [98]:
# to delete
result.test_results[0].metrics_data[0].score

0.6666666666666666

In [100]:
# to delete
result.test_results[0].metrics_data[0].reason

"The score is 0.67 because the summary introduces contradicting information regarding the authorship of 'Managing Oneself' and includes extra points that were not specified in the original text concerning self-awareness and a reader vs. listener discussion. Additionally, the summary does not address several questions that the original text could answer, highlighting a lack of completeness."

In [101]:
summary_metrics = {
    "SummarizationScore": result.test_results[0].metrics_data[0].score,
    "SummarizationReason": result.test_results[0].metrics_data[0].reason,
    "CoherenceScore": result.test_results[0].metrics_data[1].score,
    "CoherenceReason": result.test_results[0].metrics_data[1].reason,
    "ProfessionalismScore": result.test_results[0].metrics_data[2].score,
    "ProfessionalismReason": result.test_results[0].metrics_data[2].reason,
    "SafetyScore": result.test_results[0].metrics_data[3].score,
    "SafetyReason": result.test_results[0].metrics_data[3].reason,
}

In [103]:
summary_metrics
# update this to LLM output

{'SummarizationScore': 0.6666666666666666,
 'SummarizationReason': "The score is 0.67 because the summary introduces contradicting information regarding the authorship of 'Managing Oneself' and includes extra points that were not specified in the original text concerning self-awareness and a reader vs. listener discussion. Additionally, the summary does not address several questions that the original text could answer, highlighting a lack of completeness.",
 'CoherenceScore': 0.8880797085800515,
 'CoherenceReason': "The response uses clear and direct language throughout, effectively presenting complex ideas in an easy-to-follow manner. It avoids jargon, or explains it indirectly through straightforward expressions. There are no vague or confusing parts, and the content is consistent without contradictions. The summary captures the essence of Drucker's teachings, ensuring readers understand the main points regarding self-awareness and alignment with organizational values.",
 'Profession

In [ ]:
# Structured Output
from typing import Optional
from typing_extensions import Annotated, TypedDict

# Uses [] instead of () like Pydantic
class MetricsDict(TypedDict):
    SummarizationScore: Annotated[int, ..., "From 0-1"]
    SummarizationReason: Annotated[str, ..., "How well the model summarized"]
    ClarityScore: Annotated[int, ..., "Output clarity from 0-1"]

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [112]:
# Prompt for Improved Summary
system_prompt = "You are professional writer known for highly accurate."

prompt_improved = f"""
    Given the following document, do the following:

    1. Identify the book's title and author.
    2. Summarize the text in bullet points. Focus on the main ideas and key details and provide examples:

    The document is the following: 
    <document>
    {document_text}
    </document>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Summary: <summary>
"""

In [113]:
response_improved = client.responses.create(
    instructions = system_prompt,
    model = MODEL, 
    input = prompt_improved
)

In [114]:
from IPython.display import display, Markdown
display(Markdown(response_improved.output_text))

**Title:** Managing Oneself  
**Author:** Peter F. Drucker  

**Summary:**  
- **Self-Management in the Knowledge Economy:**  
  - Success requires individuals to understand their strengths, values, and best performance methods.
  - Individuals must act as their own career managers, as companies no longer manage employee careers.  
  - The working life may span nearly 50 years, necessitating ongoing engagement and productivity.

- **Key Questions for Self-Understanding:**  
  - **What Are My Strengths?**  
    - Use **feedback analysis** to identify strengths by comparing expected outcomes with actual results over time.
    - Focus on enhancing strengths rather than improving weaknesses, which typically require more effort.  
  - **How Do I Work?**  
    - Understand personal work styles: whether you process information better through reading or listening, and whether you work better alone or in groups.  
  - **What Are My Values?**  
    - Define personal ethics and ensure alignment with organizational values to avoid frustration and poor performance.  
  - **Where Do I Belong?**  
    - Find environments where one's strengths and work styles are best utilized to maximize contribution and performance.  
  - **What Can I Contribute?**  
    - Shift from being told what to do to actively determining how one can add value based on situational needs.

- **Managing Relationships:**  
  - Recognize that coworkers have their own strengths, performance modes, and values; successful work depends on understanding and adapting to these differences.  
  - Effective communication is essential to foster understanding and collaboration in organizational settings.

- **Planning for the Second Half of Life:**  
  - There’s a need to think ahead regarding career longevity, considering opportunities for second careers or parallel careers, especially as people reach midlife and seek new challenges.  
  - Engaging in parallel interests or social entrepreneurship can provide fulfillment and a sense of contribution beyond traditional work roles.

- **Final Thoughts:**  
  - Managing oneself is a critical skill for knowledge workers, requiring a proactive approach and the mindset of a chief executive officer in navigating one's career.  


In [115]:
# Original text to be summarized
input = document_text

# Summary output from LLM
actual_output = response_improved.output_text

In [116]:
test_case2 = LLMTestCase(input=input, actual_output=actual_output)

In [117]:
result2 = evaluate(test_cases=[test_case2], metrics=[metric, clarity, professionalism, safety])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                        ┃ Average Score              ┃ Pass Rate           ┃ Total       │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Summarization                                 │ 0.73                       │ 100.00%             │ 1           │
│  Clarity [GEval]                               │ 0.86                       │ 100.00%             │ 1           │
│  Professionalism [GEval]                       │ 0.92                       │ 100.00%             │ 1           │
│  Safety [GEval]                                │ 1.00                       │ 100.00%             │ 1           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=16701110;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 19.53s | token cost: 0.00535635 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [118]:
summary_metrics2 = {
    "SummarizationScore": result2.test_results[0].metrics_data[0].score,
    "SummarizationReason": result2.test_results[0].metrics_data[0].reason,
    "CoherenceScore": result2.test_results[0].metrics_data[1].score,
    "CoherenceReason": result2.test_results[0].metrics_data[1].reason,
    "ProfessionalismScore": result2.test_results[0].metrics_data[2].score,
    "ProfessionalismReason": result2.test_results[0].metrics_data[2].reason,
    "SafetyScore": result2.test_results[0].metrics_data[3].score,
    "SafetyReason": result2.test_results[0].metrics_data[3].reason,
}

In [119]:
summary_metrics2

{'SummarizationScore': 0.7333333333333333,
 'SummarizationReason': "The score is 0.73 because the summary includes multiple pieces of extra information that were not present in the original text, which may mislead readers or extend the intended message. Although the summary does not contradict the original text, the inclusion of these additional topics affects its alignment with the original content's focus.",
 'CoherenceScore': 0.8562176500885798,
 'CoherenceReason': "The response uses clear and direct language throughout the summary, effectively breaking down complex ideas into digestible sections. However, it lacks explanations for some terms such as 'feedback analysis,' which may confuse readers unfamiliar with the concept. Overall, the structured format enhances understanding, but the absence of clarification for specific jargon slightly detracts from its clarity.",
 'ProfessionalismScore': 0.9225878241117222,
 'ProfessionalismReason': "The response maintains a professional tone a

Yes I got a slightly better result, from 0.67 to 0.73. The initial evaluation indicted initially both incompletness and extra information. While the output still appears to include extra information not present in the original, the bulletpoint summary format appears to be more complete and capture the entirety of the docuement better. The output results for clarity, professionalism, and safety are around the same, with the clarity (coherence) score slightly lower. However, I am not sure if these controls are entirely enough because the fact that it still includes extra information according to the metric is a sign that this summary is not accurate and may not be taken at face value. For someone more important, like a legal document, this could have major rammifications.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
